# Prompt Engineering Demo


## Learning objectives

By the end of this demo, you should be able to:

1. Compare zero-shot and few-shot prompting.
2. Use a role to shape the style of an answer.
3. Write clear, specific instructions.
4. Separate system prompts (behaviour) from user prompts (the request).
5. Get structured JSON output you can parse in code.
6. Use temperature to control how varied the output is.

## Setup

In [1]:
# Setup
!pip install -q openai

## 1. Add your OpenAI API key as a Colab secret (one time only)

Colab **Secrets** store your key safely, so you don't have to type it every time you run the notebook.

1. Click the **key icon** ("Secrets") in the left sidebar.
2. Click **+ Add new secret**.
3. **Name:** `OPENAI_API_KEY` (exactly this, in capitals). **Value:** your OpenAI API key.
4. Turn on **Notebook access** for this notebook.

Your secrets are saved to your Google account, so the same secret works in your other Colab notebooks too (just turn on Notebook access for each one).

> Never paste your API key directly into a code cell: it would be saved with the notebook and visible to anyone you share it with.

In [2]:
import os
import json
from openai import OpenAI

def load_openai_key():
    """Read the OpenAI API key from Colab Secrets; ask for it only if that fails."""
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        print("Not running in Colab.")
    except Exception as e:
        # e.g. the secret is missing, or Notebook access is turned off
        print(f"Could not read the Colab secret OPENAI_API_KEY ({type(e).__name__}).")
        print("Check the steps above: secret name, value, and Notebook access.")
    from getpass import getpass
    return getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_openai_key()
client = OpenAI()

# Use a non-reasoning model for this demo:
# reasoning models do not accept the temperature setting used in section 7.
MODEL = "gpt-4o-mini"
print("OpenAI client is ready. Model:", MODEL)

OpenAI client is ready. Model: gpt-4o-mini


### Helper function

`ask()` sends one prompt to the model and returns the text answer. An optional `system` prompt sets the model's behaviour.

In [3]:
def ask(prompt, system=None, temperature=0.7, model=MODEL):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


print(ask("Hi"))

Hello! How can I assist you today?


## 2. Zero-shot vs few-shot

**Zero-shot:** ask directly, with no examples.
**Few-shot:** show a few examples first, so the model copies the pattern and format.

In [4]:
# Zero-shot: a direct question
prompt = "Classify this text sentiment: 'The movie was absolutely fantastic!'"

print("Zero-shot result:")
print(ask(prompt))

Zero-shot result:
The sentiment of the text "The movie was absolutely fantastic!" is positive.


In [7]:
# Few-shot: the examples fix both the labels and the format of the answer
prompt = """Classify the sentiment:

Text: "I love this product!"
Sentiment: Positive

Text: "This is terrible."
Sentiment: Negative

Text: "It's okay, nothing special."
Sentiment: Neutral

Text: "The movie was absolutely fantastic!"
Sentiment:"""

print("Few-shot result:")
print(ask(prompt))

Few-shot result:
Sentiment: Happy


**Discuss:** the zero-shot answer is a sentence; the few-shot answer is a single label. Which one is easier for a program to use?

## 3. Role prompting

A role changes the level, tone and style of the answer. Both prompts below ask for the same length, so only the role differs.

In [ ]:
# Without a role
print(ask("Explain quantum computing in about 80 words."))

Quantum computing is a revolutionary technology that leverages the principles of quantum mechanics to process information. Unlike classical computers, which use bits as the smallest unit of data (0 or 1), quantum computers utilize qubits, which can exist in multiple states simultaneously due to superposition. This allows them to perform complex calculations at unprecedented speeds. Additionally, phenomena like entanglement enable qubits to be interconnected, enhancing computational power. Quantum computing has the potential to solve problems in cryptography, optimization, and material science far beyond current capabilities.


In [ ]:
# With a role
print(ask("You are a teacher explaining to a 10-year-old. Explain quantum computing in about 80 words."))

Okay! Imagine your regular computer is like a really fast librarian who can only look at one book at a time. Quantum computing is like having a super-smart librarian who can look at many books all at once! This is because it uses tiny particles, like atoms, that can be in many states at the same time. This special ability helps quantum computers solve really complicated problems much faster than regular computers. It’s like having a magic tool for solving puzzles!


## 4. Clear instructions

Vague prompts get generic answers. Say exactly what you want: how many items, what to include, and how long.

In [ ]:
# Vague instruction
print(ask("Tell me about Paris. Where should I go?"))

Paris, the capital of France, is renowned for its art, fashion, gastronomy, and culture. Known as "The City of Light," it offers a rich tapestry of history and modernity. Here are some must-visit places and experiences in Paris:

### Iconic Landmarks
1. **Eiffel Tower**: No trip to Paris is complete without a visit to this iconic structure. You can take an elevator or climb the stairs for panoramic views of the city.
2. **Louvre Museum**: Home to thousands of works of art, including the Mona Lisa and the Venus de Milo, the Louvre is the world's largest art museum.
3. **Notre-Dame Cathedral**: Although it was damaged by fire in 2019, the cathedral remains a symbol of Paris. You can still admire its stunning architecture from the outside and explore the surrounding Île de la Cité.
4. **Arc de Triomphe**: Another iconic monument, it honors those who fought and died for France. Climb to the top for a great view of the Champs-Élysées.

### Neighborhoods to Explore
1. **Montmartre**: Famous 

In [ ]:
# Clear instruction
print(ask("List exactly 3 tourist attractions in Paris, with a one-sentence description each."))

1. **Eiffel Tower**: An iconic symbol of Paris, this iron lattice tower offers breathtaking views of the city from its observation decks.  
2. **Louvre Museum**: Home to thousands of works of art, including the Mona Lisa, this historic palace is one of the world's largest and most visited art museums.  
3. **Notre-Dame Cathedral**: A masterpiece of French Gothic architecture, this cathedral is renowned for its stunning façade, intricate stained glass, and historical significance, despite the recent fire damage.


## 5. System prompt vs user prompt

- **System prompt:** sets the behaviour for the whole conversation (the "job description").
- **User prompt:** the specific request each time.

The same system prompt applies to two different questions.

In [ ]:
system = "You are a pirate. Always respond in pirate speak, in no more than 3 sentences."

for question in ["What's the weather like?", "How do I cook pasta?"]:
    print("Question:", question)
    print("Response:", ask(question, system=system))
    print()

Question: What's the weather like?
Response: Arrr, me hearty! The skies be clear 'n the seas be calm, perfect fer plunderin'! Keep yer spyglass handy, fer a storm could brew quick as a wink!

Question: How do I cook pasta?
Response: Arrr, first ye be boilin' a pot o' salted water like the briny sea! Once it be a-boilin’, toss in yer pasta and let it cook till it be al dente, just like a true sailor’s chew! Drain it, add yer favorite sauce, and feast like the captain ye be!



## 6. Structured output (JSON)

Applications need output they can parse. Asking for JSON in plain words often works, but models sometimes wrap it in a code fence or add a sentence, and then parsing fails.

**6a. Prompt only** The prompt tells to return JSON.

In [8]:
text = "John Smith, 28 years old, works as a software engineer in San Francisco."

# 6a. Ask for JSON in plain words only
result = ask(f"""Extract the name, age, job and city from this text and return them as JSON:

"{text}"
""")
print(result)

try:
    data = json.loads(result)
    print("\n✅ Valid JSON parsed:", data)
except json.JSONDecodeError as e:
    print("\n❌ Could not parse as JSON:", e)

```json
{
  "name": "John Smith",
  "age": 28,
  "job": "software engineer",
  "city": "San Francisco"
}
```

❌ Could not parse as JSON: Expecting value: line 1 column 1 (char 0)


**6b. JSON mode** tells the API to return valid JSON only. The prompt must still mention JSON and describe the fields you want.

In [12]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract details from the text. Return JSON with the keys: name, age, job, city."},
        {"role": "user", "content": text},
    ],
    response_format={"type": "json_object"},
)

data = json.loads(response.choices[0].message.content)
print("✅ Parsed:", data)
print("Age:", data["age"])
print(data)

✅ Parsed: {'name': 'John Smith', 'age': 28, 'job': 'software engineer', 'city': 'San Francisco', 'years_of_experience': None}
Age: 28
{'name': 'John Smith', 'age': 28, 'job': 'software engineer', 'city': 'San Francisco', 'years_of_experience': None}


In [10]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract details from the text. Return JSON"},
        {"role": "user", "content": text},
    ],
    response_format={"type": "json_object"},
)

data = json.loads(response.choices[0].message.content)
print("✅ Parsed:", data)
print(data)
#print(data['job'])

✅ Parsed: {'name': 'John Smith', 'age': 28, 'occupation': 'software engineer', 'location': 'San Francisco'}
{'name': 'John Smith', 'age': 28, 'occupation': 'software engineer', 'location': 'San Francisco'}


**Discuss:** JSON mode guarantees *valid JSON*, but not the *keys*. Run the above cell: could the model return `occupation` instead of `job`? Your code would then fail on `data["job"]`.

**6c. Structured outputs (JSON Schema)** go one step further: you give the exact shape, and the model must return exactly those keys and types.

Strict mode rules:
- Every key is listed in `required`, and `additionalProperties` is `False`.
- An optional value is written as "string **or null**". The model returns `null` when the text doesn't say, instead of inventing a value. Here, `email` is not in the text.

This is the same JSON Schema idea you will see later in **function calling**.

In [11]:
person_schema = {
    "type": "object",
    "properties": {
        "name":  {"type": "string"},
        "age":   {"type": "integer"},
        "job":   {"type": "string"},
        "city":  {"type": "string"},
        "email": {"type": ["string", "null"]},   # optional: null if not in the text
    },
    "required": ["name", "age", "job", "city", "email"],
    "additionalProperties": False,
}

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract the person's details from the text."},
        {"role": "user", "content": text},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "person", "schema": person_schema, "strict": True},
    },
)

message = response.choices[0].message
if message.refusal:
    print("The model refused:", message.refusal)
else:
    data = json.loads(message.content)
    print("✅ Parsed:", data)
    print("Keys:", list(data.keys()))
    print("age is an int:", isinstance(data["age"], int))
    print("email:", data["email"])

✅ Parsed: {'name': 'John Smith', 'age': 28, 'job': 'software engineer', 'city': 'San Francisco', 'email': None}
Keys: ['name', 'age', 'job', 'city', 'email']
age is an int: True
email: None



| | Prompt only (6a) | JSON mode (6b) | Structured outputs (6c) |
|---|---|---|---|
| Valid JSON | Usually | Always | Always |
| Exact keys and types | No | No | Yes |
| Use for | Quick tests | Loose JSON | Apps, agents, pipelines |

## 7. Temperature

Temperature controls how varied the output is:

- **Low (0.0–0.3):** focused and repeatable. Good for facts, extraction and classification.
- **High (0.8–1.0):** more varied. Good for brainstorming and creative text.

Run the same prompt three times at each setting and compare.

In [ ]:
prompt = "Write a one-sentence tagline for a coffee shop."

print("Low temperature (0.1):")
for i in range(3):
    print(f"{i+1}. {ask(prompt, temperature=0.1)}")

print("\nHigh temperature (1.0):")
for i in range(3):
    print(f"{i+1}. {ask(prompt, temperature=1.0)}")

Low temperature (0.1):
1. "Awaken your senses with every sip at our cozy coffee haven."
2. "Awaken your senses with every sip at our cozy coffee haven."
3. "Awaken your senses with every sip at our cozy coffee haven."

High temperature (1.0):
1. "Awaken your senses with every sip at your cozy neighborhood coffee haven."
2. "Awaken Your Senses with Every Sip!"
3. "Awaken your senses with every sip at our cozy coffee haven."


## Summary

| Technique | Use it when |
|---|---|
| Zero-shot | The task is simple and common |
| Few-shot | You need a consistent label or format (2–5 examples are usually enough) |
| Role | You want a particular level, tone or style |
| Clear instructions | You want a specific number, length or content |
| System prompt | The behaviour should apply to every request |
| Structured output | Code will read the answer: use structured outputs (JSON Schema) when exact keys matter |
| Temperature | Low for facts and extraction, higher for creative work |

## Try it yourself

1. In section 2, change the few-shot labels to **Happy / Angry / Calm**. Does the model follow?
2. In section 5, write your own system prompt for a polite customer service assistant.
3. In section 6b, add a new key `years_of_experience`, which is not in the text. What does the model return?